In [1]:
"""
处理 WallStreetBets .zst 文件
=============================
解压并提取AMC相关的posts和comments

Requirements:
    pip install zstandard pandas tqdm
"""

import zstandard as zstd
import json
import pandas as pd
from datetime import datetime
from tqdm import tqdm
import re

# ============================================================
# 设置路径
# ============================================================
SUBMISSIONS_FILE = r'D:\wallstreetbets\reddit\subreddits\wallstreetbets_submissions.zst'
COMMENTS_FILE = r'D:\wallstreetbets\reddit\subreddits\wallstreetbets_comments.zst'

# 时间范围
START_DATE = '2019-07-01'
END_DATE = '2021-06-30'

# 关键词
KEYWORDS = ['AMC', '$AMC', 'AMC Entertainment']


def read_and_process_zst_streaming(file_path, start_ts, end_ts, keywords, is_submission=True):
    """
    流式读取和处理.zst文件（避免内存溢出）
    """
    print(f"\n流式处理: {file_path}")
    
    results = []
    line_count = 0
    matched_count = 0
    
    with open(file_path, 'rb') as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as reader:
            text_reader = reader.read(1024 * 1024)  # 每次读1MB
            buffer = b''
            
            while text_reader:
                buffer += text_reader
                
                # 按行分割
                lines = buffer.split(b'\n')
                buffer = lines[-1]  # 保留最后不完整的行
                
                for line in lines[:-1]:
                    line_count += 1
                    
                    if line_count % 10000 == 0:
                        print(f"  处理: {line_count:,} 行, 匹配: {matched_count:,} 条", end='\r')
                    
                    try:
                        data = json.loads(line.decode('utf-8'))
                        
                        # 检查时间
                        created_utc = data.get('created_utc', 0)
                        if created_utc < start_ts or created_utc > end_ts:
                            continue
                        
                        # 检查关键词
                        if is_submission:
                            title = data.get('title', '').lower()
                            selftext = data.get('selftext', '').lower()
                            has_keyword = any(kw.lower() in title or kw.lower() in selftext 
                                            for kw in keywords)
                        else:
                            body = data.get('body', '').lower()
                            has_keyword = any(kw.lower() in body for kw in keywords)
                        
                        if not has_keyword:
                            continue
                        
                        # 提取数据
                        if is_submission:
                            results.append({
                                'post_id': data.get('id'),
                                'author': data.get('author', '[deleted]'),
                                'title': data.get('title', ''),
                                'selftext': data.get('selftext', ''),
                                'score': data.get('score', 0),
                                'num_comments': data.get('num_comments', 0),
                                'created_utc': created_utc,
                                'timestamp': datetime.fromtimestamp(created_utc),
                                'url': f"https://reddit.com{data.get('permalink', '')}"
                            })
                        else:
                            results.append({
                                'comment_id': data.get('id'),
                                'post_id': data.get('link_id', '').replace('t3_', ''),
                                'parent_id': data.get('parent_id', '').replace('t1_', '').replace('t3_', ''),
                                'author': data.get('author', '[deleted]'),
                                'body': data.get('body', ''),
                                'score': data.get('score', 0),
                                'created_utc': created_utc,
                                'timestamp': datetime.fromtimestamp(created_utc)
                            })
                        
                        matched_count += 1
                        
                    except:
                        continue
                
                # 读取下一块
                text_reader = reader.read(1024 * 1024)
    
    print(f"\n✓ 完成: 总共 {line_count:,} 行, 匹配 {matched_count:,} 条")
    return pd.DataFrame(results)


def process_submissions(file_path, start_date, end_date, keywords):
    """
    处理帖子文件
    """
    print("\n" + "="*60)
    print("处理帖子 (Submissions)")
    print("="*60)
    
    # 转换日期为时间戳
    start_ts = datetime.strptime(start_date, '%Y-%m-%d').timestamp()
    end_ts = datetime.strptime(end_date, '%Y-%m-%d').timestamp()
    
    # 流式处理
    df = read_and_process_zst_streaming(
        file_path, start_ts, end_ts, keywords, is_submission=True
    )
    
    print(f"\n✓ 提取到 {len(df):,} 篇AMC相关帖子")
    
    if len(df) > 0:
        print(f"  时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
        print(f"  作者数: {df['author'].nunique():,}")
    
    return df


def process_comments(file_path, start_date, end_date, keywords):
    """
    处理评论文件
    """
    print("\n" + "="*60)
    print("处理评论 (Comments)")
    print("="*60)
    
    start_ts = datetime.strptime(start_date, '%Y-%m-%d').timestamp()
    end_ts = datetime.strptime(end_date, '%Y-%m-%d').timestamp()
    
    # 流式处理
    df = read_and_process_zst_streaming(
        file_path, start_ts, end_ts, keywords, is_submission=False
    )
    
    print(f"\n✓ 提取到 {len(df):,} 条AMC相关评论")
    
    if len(df) > 0:
        print(f"  时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
        print(f"  作者数: {df['author'].nunique():,}")
    
    return df


def extract_mentions(text):
    """提取用户提及"""
    if not text or pd.isna(text):
        return []
    mentions = re.findall(r'/?u/(\w+)', str(text), re.IGNORECASE)
    return list(set(mentions))


def process_for_network_analysis(posts_df, comments_df):
    """
    转换为网络分析格式
    """
    print("\n" + "="*60)
    print("转换为网络分析格式")
    print("="*60)
    
    # 处理帖子
    posts_processed = []
    for _, post in posts_df.iterrows():
        text = f"{post['title']} {post.get('selftext', '')}"
        mentions = extract_mentions(text)
        
        posts_processed.append({
            'timestamp': post['timestamp'],
            'user_id': post['author'],
            'post_id': post['post_id'],
            'text': str(text)[:500],
            'mentions': mentions,
            'reply_to_user': None,
            'score': post['score'],
            'type': 'post'
        })
    
    # 处理评论
    comments_processed = []
    
    if len(comments_df) > 0:
        # 建立映射
        comment_authors = dict(zip(comments_df['comment_id'], comments_df['author']))
        post_authors = dict(zip(posts_df['post_id'], posts_df['author']))
        
        for _, comment in comments_df.iterrows():
            mentions = extract_mentions(comment.get('body', ''))
            
            # 确定回复对象
            parent_id = comment.get('parent_id')
            reply_to_user = None
            
            if parent_id:
                if parent_id in comment_authors:
                    reply_to_user = comment_authors[parent_id]
                elif parent_id in post_authors:
                    reply_to_user = post_authors[parent_id]
            
            comments_processed.append({
                'timestamp': comment['timestamp'],
                'user_id': comment['author'],
                'post_id': comment['comment_id'],
                'text': str(comment.get('body', ''))[:500],
                'mentions': mentions,
                'reply_to_user': reply_to_user,
                'score': comment['score'],
                'type': 'comment'
            })
    
    # 合并
    all_data = posts_processed + comments_processed
    df = pd.DataFrame(all_data)
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    print(f"\n✓ 处理完成:")
    print(f"  帖子: {len(posts_processed):,}")
    print(f"  评论: {len(comments_processed):,}")
    print(f"  总计: {len(df):,}")
    
    return df


# === 主程序 ===
if __name__ == "__main__":
    
    print("\n" + "="*60)
    print("WallStreetBets .zst 文件处理器 — AMC")
    print("="*60)
    print(f"\n时间范围: {START_DATE} ~ {END_DATE}")
    print(f"关键词: {', '.join(KEYWORDS)}")
    
    try:
        # 1. 处理帖子
        posts_df = process_submissions(
            SUBMISSIONS_FILE,
            START_DATE,
            END_DATE,
            KEYWORDS
        )
        
        # 2. 处理评论
        comments_df = process_comments(
            COMMENTS_FILE,
            START_DATE,
            END_DATE,
            KEYWORDS
        )
        
        # 3. 转换格式
        processed_df = process_for_network_analysis(posts_df, comments_df)
        
        # 4. 保存
        print("\n" + "="*60)
        print("保存数据")
        print("="*60)
        
        posts_df.to_csv('reddit_wsb_amc_posts_raw.csv', index=False)
        comments_df.to_csv('reddit_wsb_amc_comments_raw.csv', index=False)
        processed_df.to_csv('reddit_wsb_amc_for_network.csv', index=False)
        
        print("✓ 已保存:")
        print("  - reddit_wsb_posts_raw.csv")
        print("  - reddit_wsb_comments_raw.csv")
        print("  - reddit_wsb_for_network.csv")
        
        # 统计
        print(f"\n" + "="*60)
        print("数据统计")
        print("="*60)
        print(f"帖子: {len(posts_df):,}")
        print(f"评论: {len(comments_df):,}")
        print(f"唯一用户: {processed_df['user_id'].nunique():,}")
        print(f"总提及: {sum(len(m) for m in processed_df['mentions']):,}")
        print(f"总回复: {processed_df['reply_to_user'].notna().sum():,}")
        print("="*60)
        
        print("\n✓ 完成！")
        print("下一步: 使用 reddit_wsb_amc_for_network.csv 进行网络分析")
        
    except Exception as e:
        print(f"\n✗ 错误: {e}")
        import traceback
        traceback.print_exc()


WallStreetBets .zst 文件处理器 — AMC

时间范围: 2019-07-01 ~ 2021-06-30
关键词: AMC, $AMC, AMC Entertainment

处理帖子 (Submissions)

流式处理: D:\wallstreetbets\reddit\subreddits\wallstreetbets_submissions.zst
  处理: 2,210,000 行, 匹配: 98,560 条
✓ 完成: 总共 2,218,243 行, 匹配 98,560 条

✓ 提取到 98,560 篇AMC相关帖子
  时间范围: 2019-07-03 21:51:33 ~ 2021-06-29 23:19:19
  作者数: 57,306

处理评论 (Comments)

流式处理: D:\wallstreetbets\reddit\subreddits\wallstreetbets_comments.zst
  处理: 69,620,000 行, 匹配: 477,424 条
✓ 完成: 总共 69,623,416 行, 匹配 477,424 条

✓ 提取到 477,424 条AMC相关评论
  时间范围: 2019-07-01 02:32:01 ~ 2021-06-29 23:58:09
  作者数: 156,897

转换为网络分析格式

✓ 处理完成:
  帖子: 98,560
  评论: 477,424
  总计: 575,984

保存数据
✓ 已保存:
  - reddit_wsb_posts_raw.csv
  - reddit_wsb_comments_raw.csv
  - reddit_wsb_for_network.csv

数据统计
帖子: 98,560
评论: 477,424
唯一用户: 200,353
总提及: 5,742
总回复: 111,712

✓ 完成！
下一步: 使用 reddit_wsb_amc_for_network.csv 进行网络分析
